In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
#storage_details
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<YOUR KEY>"
)


In [0]:
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/products"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dimensions/dim_products"

In [0]:
df = spark.read.format("delta").load(silver_path)

In [0]:
required_columns = [
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

In [0]:
missing_columns = [c for c in required_columns
                if c not in df.columns]

if missing_columns:
    raise Exception(f"Missing columns: {missing_columns}")

In [0]:
dim_columns = (
    df.select(required_columns)
    .dropDuplicates(["product_id"])
    .withColumn("product_key", F.monotonically_increasing_id())
)


In [0]:
dim_columns.write.format("delta").save(gold_path)